# Final pipeline
Train/test split first, then BGMM routing: XGBoost on the 2 best clusters, expert ensemble on the rest.

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import xgboost as xgb
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.cluster import AgglomerativeClustering, HDBSCAN
from sklearn.metrics import average_precision_score, classification_report, confusion_matrix, precision_recall_curve
from sklearn.metrics.pairwise import laplacian_kernel
from sklearn.mixture import BayesianGaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.svm import SVC

warnings.filterwarnings('ignore')

In [ ]:
def report(y_true, y_pred, scores=None):
    print(confusion_matrix(y_true, y_pred))
    print(classification_report(y_true, y_pred, digits=3))
    if scores is not None:
        print(f"AUPRC: {average_precision_score(y_true, scores):.4f}")


def best_threshold(y_true, scores):
    prec, rec, thresh = precision_recall_curve(y_true, scores)
    f1 = 2 * prec * rec / (prec + rec + 1e-8)
    return thresh[np.argmax(f1[:-1])]

## Data

In [ ]:
data = pd.read_csv('../dataset/creditcard.csv').drop(['Time'], axis=1)
y = data['Class']
X = data.drop(['Class'], axis=1)
X['Amount'] = np.log(X['Amount'] + 0.001)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

amount_scaler = RobustScaler().fit(X_train[['Amount']])
X_train['Amount'] = amount_scaler.transform(X_train[['Amount']])
X_test['Amount'] = amount_scaler.transform(X_test[['Amount']])

## BGMM routing (fit on train only)

In [ ]:
model_path = '../models/bgmm_train.joblib'

if os.path.exists(model_path):
    bgm = joblib.load(model_path)
else:
    bgm = BayesianGaussianMixture(
        n_components=15,
        weight_concentration_prior_type='dirichlet_process',
        weight_concentration_prior=1e-3,
        max_iter=500,
        random_state=42,
        verbose=1
    )
    bgm.fit(X_train)
    joblib.dump(bgm, model_path)

train_clusters = bgm.predict(X_train)
test_clusters = bgm.predict(X_test)

crosstab = pd.crosstab(train_clusters, y_train.values).rename(columns={0: 'Normal', 1: 'Fraud'})
crosstab['Fraud_Rate (%)'] = crosstab['Fraud'] / crosstab.sum(axis=1) * 100
crosstab = crosstab.sort_values('Fraud_Rate (%)', ascending=False)
print(crosstab)

best_clusters = crosstab.index[:2]
print(f"Best clusters: {list(best_clusters)}")

In [ ]:
easy_train = np.isin(train_clusters, best_clusters)
easy_test = np.isin(test_clusters, best_clusters)

X_easy_train, y_easy_train = X_train[easy_train], y_train[easy_train]
X_easy_test, y_easy_test = X_test[easy_test], y_test[easy_test]
X_hard_train, y_hard_train = X_train[~easy_train], y_train[~easy_train]
X_hard_test, y_hard_test = X_test[~easy_test], y_test[~easy_test]

print(f"Easy: {len(X_easy_train)} train ({y_easy_train.sum()} fraud), {len(X_easy_test)} test ({y_easy_test.sum()} fraud)")
print(f"Hard: {len(X_hard_train)} train ({y_hard_train.sum()} fraud), {len(X_hard_test)} test ({y_hard_test.sum()} fraud)")

## XGBoost on the best clusters

In [ ]:
xgb_model = xgb.XGBClassifier(
    scale_pos_weight=(y_easy_train == 0).sum() / (y_easy_train == 1).sum(),
    max_depth=4,
    learning_rate=0.05,
    n_estimators=300,
    objective='binary:logistic',
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_easy_train, y_easy_train)

easy_scores = xgb_model.predict_proba(X_easy_test)[:, 1]
easy_pred = (easy_scores >= 0.5).astype(int)
report(y_easy_test, easy_pred, easy_scores)

## Expert architecture on the rest
The hard train set is split again into fit/val. Val is used for TabNet early stopping and for picking thresholds, so the test set is only touched for the final evaluation.

In [ ]:
X_fit, X_val, y_fit, y_val = train_test_split(X_hard_train, y_hard_train, test_size=0.2, random_state=42, stratify=y_hard_train)

scaler = StandardScaler().fit(X_fit)
X_fit_s, X_val_s, X_test_s = (scaler.transform(d).astype(np.float32) for d in (X_fit, X_val, X_hard_test))
X0_s = X_fit_s[y_fit.values == 0]

frauds = X_fit[y_fit.values == 1]
frauds_s = X_fit_s[y_fit.values == 1]

### Assigning frauds to experts
The three clusterings are combined with a co-association matrix (fraction of methods that put two frauds together) instead of hand-written rules, since cluster ids change on every refit. Experts with fewer than 3 frauds are merged into one leftover expert.

In [ ]:
N_EXPERTS = 7

labels = [
    BayesianGaussianMixture(n_components=10, covariance_type='diag', weight_concentration_prior=10.0, max_iter=500, random_state=42).fit_predict(frauds),
    AgglomerativeClustering(n_clusters=None, distance_threshold=0.65, metric='cosine', linkage='average').fit_predict(frauds),
    HDBSCAN(min_cluster_size=3, min_samples=2).fit_predict(frauds),
]
co_assoc = np.mean([(l[:, None] == l[None, :]) & (l[:, None] != -1) for l in labels], axis=0)
np.fill_diagonal(co_assoc, 1)

expert_labels = AgglomerativeClustering(n_clusters=N_EXPERTS, metric='precomputed', linkage='average').fit_predict(1 - co_assoc)
expert_labels[np.bincount(expert_labels)[expert_labels] < 3] = N_EXPERTS
expert_labels = np.unique(expert_labels, return_inverse=True)[1]

fraud_groups = [frauds_s[expert_labels == i] for i in range(expert_labels.max() + 1)]
print(f"Frauds per expert: {[len(g) for g in fraud_groups]}")

### Hard Negative SVM experts

In [ ]:
def hn_svm_expert(X_fraud, rng, eval_sets):
    fisher_w = np.abs(X_fraud.mean(axis=0)) / (X_fraud.std(axis=0) + 0.5)
    fisher_w = np.clip(fisher_w / (fisher_w.mean() + 1e-8), 0.2, 3.0)
    X0_w, X_fraud_w = X0_s * fisher_w, X_fraud * fisher_w

    rand_idx = rng.choice(len(X0_w), size=2000, replace=False)
    X_p1 = np.vstack([X0_w[rand_idx], X_fraud_w])
    y_p1 = np.r_[np.zeros(len(rand_idx)), np.ones(len(X_fraud_w))]
    svm_p1 = SVC(kernel='rbf', C=2.0, class_weight='balanced').fit(X_p1, y_p1)

    hardest_idx = np.argsort(svm_p1.decision_function(X0_w))[-1000:]
    neg_idx = np.unique(np.r_[rand_idx, hardest_idx])
    X_p2 = np.vstack([X0_w[neg_idx], X_fraud_w])
    y_p2 = np.r_[np.zeros(len(neg_idx)), np.ones(len(X_fraud_w))]

    svm_rbf = SVC(kernel='rbf', C=3.0, gamma='scale', class_weight='balanced', probability=True, random_state=42).fit(X_p2, y_p2)
    gamma_lap = 1.0 / X_p2.shape[1]
    svm_lap = SVC(kernel='precomputed', C=3.0, class_weight='balanced', probability=True, random_state=42)
    svm_lap.fit(laplacian_kernel(X_p2, X_p2, gamma=gamma_lap), y_p2)

    return [
        0.5 * svm_rbf.predict_proba(X * fisher_w)[:, 1]
        + 0.5 * svm_lap.predict_proba(laplacian_kernel(X * fisher_w, X_p2, gamma=gamma_lap))[:, 1]
        for X in eval_sets
    ]


rng = np.random.RandomState(42)
svm_preds = [hn_svm_expert(g, rng, [X_val_s, X_test_s]) for g in fraud_groups]
svm_val = np.max([p[0] for p in svm_preds], axis=0)
svm_test = np.max([p[1] for p in svm_preds], axis=0)

### TabNet experts

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'


def tabnet_expert(X_fraud, eval_sets):
    X_tr = np.vstack([X0_s, X_fraud])
    y_tr = np.r_[np.zeros(len(X0_s)), np.ones(len(X_fraud))].astype(int)
    weights = torch.tensor([1.0, np.sqrt(len(X0_s) / len(X_fraud))], dtype=torch.float32).to(device)

    clf = TabNetClassifier(
        n_d=16, n_a=16, n_steps=3, gamma=1.3, lambda_sparse=1e-3,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2, weight_decay=1e-5),
        mask_type='sparsemax',
        device_name=device,
        verbose=0,
        seed=42
    )
    clf.fit(
        X_train=X_tr, y_train=y_tr,
        eval_set=[(X_val_s, y_val.values)],
        eval_metric=['logloss'],
        loss_fn=nn.CrossEntropyLoss(weight=weights),
        max_epochs=40, patience=10,
        batch_size=8192, virtual_batch_size=512,
        drop_last=False
    )
    return [clf.predict_proba(X)[:, 1] for X in eval_sets]


tabnet_preds = [tabnet_expert(g, [X_val_s, X_test_s]) for g in fraud_groups]
tabnet_val = np.max([p[0] for p in tabnet_preds], axis=0)
tabnet_test = np.max([p[1] for p in tabnet_preds], axis=0)

## Results
### Hard clusters only

In [ ]:
experts = {'HN-SVM': (svm_val, svm_test), 'TabNet': (tabnet_val, tabnet_test)}
thresholds = {}

for name, (val_scores, test_scores) in experts.items():
    thresholds[name] = best_threshold(y_val, val_scores)
    print(f"--- {name} (threshold {thresholds[name]:.4f} from val) ---")
    report(y_hard_test, (test_scores >= thresholds[name]).astype(int), test_scores)

### Full pipeline on the whole test set

In [ ]:
for name, (_, test_scores) in experts.items():
    y_pred = np.zeros(len(X_test), dtype=int)
    y_pred[easy_test] = easy_pred
    y_pred[~easy_test] = test_scores >= thresholds[name]
    print(f"--- XGBoost + {name} ---")
    report(y_test, y_pred)